In [ ]:
import os
import torch
import torch.backends.cudnn as cudnn
from torch import nn
from easydict import EasyDict as edict
from models import Generator, Discriminator, TruncatedVGG19
from datasets import SRDataset
from utils import *
from solver import pretrain_generator, train_gan, EMA

# ========================================
# CONFIGURATION
# ========================================
config = edict()

# Data paths
config.csv_folder = "data"
config.HR_data_folder = "data/DIV2K_train_HR"
config.LR_data_folder = "data/DIV2K_train_LR_bicubic/X4"

# Data parameters
config.crop_size = 96
config.scaling_factor = 4

# Generator parameters (STRICT paper compliance)
config.G = edict()
config.G.large_kernel_size = 9
config.G.small_kernel_size = 3
config.G.n_channels = 64
config.G.n_blocks = 16

# Discriminator parameters
config.D = edict()
config.D.kernel_size = 3
config.D.n_channels = 64
config.D.n_blocks = 8
config.D.fc_size = 1024

# Spectral Normalization
config.use_spectral_norm = True

# Training strategy
config.checkpoint = None

# Stage 1: Generator pre-training
config.pretrain_epochs = 30
config.lr_pretrain = 1e-4
config.weight_decay_pretrain = 1e-4

# 🔥 FIX 1: Stage 2 学习率分离配置
config.train_epochs = 200
config.lr_g = 1e-4  # 生成器学习率
config.lr_d = 5e-6  # 🔥 判别器学习率（降低20倍！）
config.weight_decay_g = 1e-4
config.weight_decay_d = 1e-3
config.start_epoch = 0

# Optimization settings
config.batch_size = 16
config.workers = 16

# 🔥 FIX 2: 增加对抗损失权重
config.beta = 0.005  # 🔥 从 1e-3 增加到 5e-3

# Learning rate decay
config.lr_decay_factor = 0.5
config.lr_decay_epochs = 100

# Training tricks
config.grad_clip = 1.0
config.use_amp = False

config.warmup_epochs = 5

# 🔥 FIX 3: 优化标签平滑和噪声
config.use_label_smoothing = True
config.real_label_smoothing = 0.85  # 🔥 从 0.9 降到 0.85
config.label_noise_prob = 0.1  # 🔥 从 0.05 增加到 0.1

config.use_gradient_penalty = True
config.lambda_gp = 20.0  # 🔥 从 10.0 增加到 20.0

config.use_ema = True
config.ema_decay = 0.9999

config.use_augmentation = True

config.use_vgg_pretrain = True
config.vgg_weight_pretrain = 0.2
config.use_multilayer_vgg = True
config.use_style_loss = False

config.vgg19_i = 5
config.vgg19_j = 4
config.vgg19_i_low = 2
config.vgg19_j_low = 2

config.vgg_weight_low = 0.2
config.vgg_weight_high = 0.8

config.style_weight = 0.1

# Gradient accumulation
config.gradient_accumulation = 1

# Logging and checkpointing
config.print_freq = 50
config.save_freq = 10

# Device
config.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cudnn.benchmark = True

# ========================================
# PRINT CONFIGURATION
# ========================================
print("\n" + "=" * 80)
print("SRGAN TRAINING CONFIGURATION (FIXED BALANCED VERSION)")
print("=" * 80)
print(f"Device: {config.device}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

print("\n✅ Optimizer: AdamW (decoupled weight decay)")
print(f"Batch Size: {config.batch_size}")
print(f"Workers: {config.workers}")
print(f"Gradient Accumulation: {config.gradient_accumulation}")
print(f"Data Augmentation: {'Enabled' if config.use_augmentation else 'Disabled'}")

print("\n📊 Stage 1 (Pre-training):")
print(f"  Epochs: {config.pretrain_epochs}")
print(f"  Learning Rate: {config.lr_pretrain}")
print(f"  Weight Decay: {config.weight_decay_pretrain}")
print(f"  Warmup Epochs: {config.warmup_epochs}")
print(
    f"  VGG Loss: {'Enabled' if config.use_vgg_pretrain else 'Disabled'} ({config.vgg_weight_pretrain * 100:.0f}%)"
)

print("\n🎯 Stage 2 (GAN Training):")
print(f"  Epochs: {config.train_epochs}")
print(f"  🔥 Generator LR: {config.lr_g}")
print(f"  🔥 Discriminator LR: {config.lr_d} (20x lower!)")  # 🔥 突出显示
print(f"  Generator Weight Decay: {config.weight_decay_g}")
print(f"  Discriminator Weight Decay: {config.weight_decay_d}")
print(f"  🔥 Beta (Adversarial Weight): {config.beta} (5x increased!)")  # 🔥 突出显示

print("\n🔧 Training Tricks:")
print(f"  Gradient Clipping: {config.grad_clip}")
print(f"  Mixed Precision (AMP): {config.use_amp}")
print(
    f"  🔥 Label Smoothing: {config.use_label_smoothing} (Real={config.real_label_smoothing})"
)
print(f"  🔥 Label Noise: {config.label_noise_prob * 100:.0f}%")
print(
    f"  🔥 Gradient Penalty (R1): {config.use_gradient_penalty} (λ={config.lambda_gp})"
)
print(f"  EMA: {config.use_ema} (decay={config.ema_decay})")
print(
    f"  Multi-layer VGG: {config.use_multilayer_vgg} (Low={config.vgg_weight_low}, High={config.vgg_weight_high})"
)
print(
    f"  Style Loss: {config.use_style_loss}"
    + (f" (weight={config.style_weight})" if config.use_style_loss else "")
)
print(f"  Spectral Normalization: {config.use_spectral_norm}")

print("=" * 80 + "\n")

# ========================================
# INITIALIZE MODELS
# ========================================
if config.checkpoint is None:
    print("🔨 Initializing models from scratch...\n")

    generator = Generator(config)
    discriminator = Discriminator(config)

    if hasattr(discriminator, "use_spectral_norm"):
        print(
            f"✅ Discriminator Spectral Normalization: {'Enabled' if discriminator.use_spectral_norm else 'Disabled'}"
        )

    optimizer_g_pretrain = torch.optim.AdamW(
        generator.parameters(),
        lr=config.lr_pretrain,
        betas=(0.9, 0.999),
        weight_decay=config.weight_decay_pretrain,
        eps=1e-8,
    )

    # 🔥 FIX 4: 生成器使用 lr_g
    optimizer_g = torch.optim.AdamW(
        generator.parameters(),
        lr=config.lr_g,  # 🔥 使用独立配置
        betas=(0.9, 0.999),
        weight_decay=config.weight_decay_g,
        eps=1e-8,
    )

    # 🔥 FIX 5: 判别器使用 lr_d（远小于生成器）
    optimizer_d = torch.optim.AdamW(
        discriminator.parameters(),
        lr=config.lr_d,  # 🔥 关键：使用更低的学习率
        betas=(0.9, 0.999),
        weight_decay=config.weight_decay_d,
        eps=1e-8,
    )

    start_pretrain_epoch = 0
    start_gan_epoch = 0

    ema = EMA(generator, decay=config.ema_decay) if config.use_ema else None
    if ema:
        print(f"✅ EMA initialized with decay: {config.ema_decay}\n")

else:
    print(f"📂 Loading checkpoint: {config.checkpoint}\n")
    checkpoint = torch.load(config.checkpoint, map_location="cpu")

    generator = checkpoint["generator"]
    discriminator = checkpoint["discriminator"]

    optimizer_g_pretrain = checkpoint.get("optimizer_g_pretrain", None)
    optimizer_g = checkpoint.get("optimizer_g", None)
    optimizer_d = checkpoint.get("optimizer_d", None)

    if optimizer_g_pretrain is None:
        print("⚠️ optimizer_g_pretrain not found, rebuilding...")
        optimizer_g_pretrain = torch.optim.AdamW(
            generator.parameters(),
            lr=config.lr_pretrain,
            betas=(0.9, 0.999),
            weight_decay=config.weight_decay_pretrain,
            eps=1e-8,
        )

    if optimizer_g is None:
        print("⚠️ optimizer_g not found, rebuilding with AdamW...")
        optimizer_g = torch.optim.AdamW(
            generator.parameters(),
            lr=config.lr_g,  # 🔥 使用 lr_g
            betas=(0.9, 0.999),
            weight_decay=config.weight_decay_g,
            eps=1e-8,
        )

    if optimizer_d is None:
        print("⚠️ optimizer_d not found, rebuilding with AdamW...")
        optimizer_d = torch.optim.AdamW(
            discriminator.parameters(),
            lr=config.lr_d,  # 🔥 使用 lr_d
            betas=(0.9, 0.999),
            weight_decay=config.weight_decay_d,
            eps=1e-8,
        )

    start_pretrain_epoch = checkpoint.get("pretrain_epoch", config.pretrain_epochs)
    start_gan_epoch = checkpoint.get("gan_epoch", 0)

    if config.use_ema and "ema_shadow" in checkpoint:
        ema = EMA(generator, decay=config.ema_decay)
        ema.shadow = checkpoint["ema_shadow"]
        print("✅ Loaded EMA weights from checkpoint")
    else:
        ema = EMA(generator, decay=config.ema_decay) if config.use_ema else None

    print(
        f"✅ Resumed: Pretrain epoch {start_pretrain_epoch}, GAN epoch {start_gan_epoch}\n"
    )

# ========================================
# VGG19 FOR PERCEPTUAL LOSS
# ========================================
print("📥 Loading VGG19 for perceptual loss...")
truncated_vgg19 = TruncatedVGG19(i=config.vgg19_i, j=config.vgg19_j)
truncated_vgg19.eval()
for param in truncated_vgg19.parameters():
    param.requires_grad = False

if config.use_multilayer_vgg:
    truncated_vgg19_low = TruncatedVGG19(i=config.vgg19_i_low, j=config.vgg19_j_low)
    truncated_vgg19_low.eval()
    for param in truncated_vgg19_low.parameters():
        param.requires_grad = False
    print(
        f"✅ Multi-layer VGG enabled: φ{config.vgg19_i},{config.vgg19_j} + φ{config.vgg19_i_low},{config.vgg19_j_low}"
    )
else:
    truncated_vgg19_low = None

# Loss functions
content_loss_criterion = nn.MSELoss()
adversarial_loss_criterion = nn.BCEWithLogitsLoss()

# Move to device
generator = generator.to(config.device)
discriminator = discriminator.to(config.device)
truncated_vgg19 = truncated_vgg19.to(config.device)
if truncated_vgg19_low:
    truncated_vgg19_low = truncated_vgg19_low.to(config.device)
content_loss_criterion = content_loss_criterion.to(config.device)
adversarial_loss_criterion = adversarial_loss_criterion.to(config.device)

print(f"✅ Generator parameters: {sum(p.numel() for p in generator.parameters()):,}")
print(
    f"✅ Discriminator parameters: {sum(p.numel() for p in discriminator.parameters()):,}\n"
)

# Learning rate schedulers
scheduler_g = torch.optim.lr_scheduler.StepLR(
    optimizer_g, step_size=config.lr_decay_epochs, gamma=config.lr_decay_factor
)

scheduler_d = torch.optim.lr_scheduler.StepLR(
    optimizer_d, step_size=config.lr_decay_epochs, gamma=config.lr_decay_factor
)

config.truncated_vgg19_low = truncated_vgg19_low

# ========================================
# DATA LOADER
# ========================================
print("📦 Loading training dataset...")
train_dataset = SRDataset(
    split="train",
    config=config,
)
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.workers,
    pin_memory=True,
)

print(f"✅ Training dataset: {len(train_dataset)} samples")
print(f"✅ Batches per epoch: {len(train_loader)}\n")

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("logs", exist_ok=True)

# ========================================
# STAGE 1: PRE-TRAIN GENERATOR
# ========================================
if start_pretrain_epoch < config.pretrain_epochs:
    print("=" * 80)
    print("🚀 STAGE 1: PRE-TRAINING GENERATOR")
    print("=" * 80)
    print(
        "Loss: MSE (80%) + VGG (20%)" if config.use_vgg_pretrain else "Loss: Pure MSE"
    )
    print(f"Optimizer: AdamW (weight_decay={config.weight_decay_pretrain})")
    print(f"Learning rate warmup: first {config.warmup_epochs} epochs")
    if ema:
        print(f"EMA enabled: decay={config.ema_decay}")
    print("=" * 80 + "\n")

    best_pretrain_loss = float("inf")

    for epoch in range(start_pretrain_epoch, config.pretrain_epochs):
        print(f"\n{'=' * 80}")
        print(f"📊 Pretrain Epoch [{epoch + 1}/{config.pretrain_epochs}]")
        print(f"Learning Rate: {optimizer_g_pretrain.param_groups[0]['lr']:.6f}")
        print(f"{'=' * 80}\n")

        mse_loss = pretrain_generator(
            train_loader=train_loader,
            generator=generator,
            optimizer_g=optimizer_g_pretrain,
            epoch=epoch,
            device=config.device,
            config=config,
            ema=ema,
        )

        print(f"\n{'=' * 80}")
        print(f"✅ Pretrain Epoch [{epoch + 1}/{config.pretrain_epochs}] Complete")
        print(f"Average Loss: {mse_loss:.6f}")
        print(f"{'=' * 80}\n")

        if mse_loss < best_pretrain_loss:
            best_pretrain_loss = mse_loss
            print(f"🏆 New best loss: {mse_loss:.6f} - Saving checkpoint...")

            save_dict = {
                "pretrain_epoch": epoch + 1,
                "generator": generator,
                "optimizer_g_pretrain": optimizer_g_pretrain,
                "mse_loss": mse_loss,
            }

            if ema is not None:
                save_dict["ema_shadow"] = ema.shadow

            save_checkpoint(
                save_dict, filename="checkpoints/checkpoint_srgan_pretrain_best.pth.tar"
            )
            print()

        if (epoch + 1) % config.save_freq == 0:
            checkpoint_path = (
                f"checkpoints/checkpoint_srgan_pretrain_epoch{epoch + 1}.pth.tar"
            )

            save_dict = {
                "pretrain_epoch": epoch + 1,
                "generator": generator,
                "optimizer_g_pretrain": optimizer_g_pretrain,
            }

            if ema is not None:
                save_dict["ema_shadow"] = ema.shadow

            save_checkpoint(save_dict, filename=checkpoint_path)
            print(f"💾 Checkpoint saved: {checkpoint_path}\n")

    print("\n" + "=" * 80)
    print("💾 SAVING FINAL PRETRAIN CHECKPOINT")
    print("=" * 80)

    save_dict = {
        "pretrain_epoch": config.pretrain_epochs,
        "generator": generator,
        "optimizer_g_pretrain": optimizer_g_pretrain,
    }

    if ema is not None:
        save_dict["ema_shadow"] = ema.shadow

    save_checkpoint(
        save_dict, filename="checkpoints/checkpoint_srgan_pretrain_final.pth.tar"
    )

    print("\n✅ Pre-training Complete!")
    print(f"🏆 Best Loss: {best_pretrain_loss:.6f}")
    print("=" * 80 + "\n")

# ========================================
# STAGE 2: GAN ADVERSARIAL TRAINING
# ========================================
print("=" * 80)
print("🎯 STAGE 2: GAN ADVERSARIAL TRAINING (BALANCED VERSION)")
print("=" * 80)
print("Loss: Content (VGG φ5,4) + Adversarial")
print(f"Optimizer: AdamW (G_LR={config.lr_g}, D_LR={config.lr_d})")
print(f"🔥 D/G LR Ratio: {config.lr_d / config.lr_g:.1f}x (discriminator slower)")
if config.use_label_smoothing:
    print(
        f"Label smoothing: Real={config.real_label_smoothing}, Noise={config.label_noise_prob}"
    )
if config.use_gradient_penalty:
    print(f"Gradient penalty (R1): λ={config.lambda_gp}")
if config.use_multilayer_vgg:
    print(
        f"Multi-layer VGG: φ{config.vgg19_i},{config.vgg19_j} ({config.vgg_weight_high * 100:.0f}%) + φ{config.vgg19_i_low},{config.vgg19_j_low} ({config.vgg_weight_low * 100:.0f}%)"
    )
if ema:
    print(f"EMA enabled: decay={config.ema_decay}")
print("=" * 80 + "\n")


# 🔥 FIX 6: 添加训练监控函数
def monitor_training_health(epoch, metrics):
    """监控训练健康度并给出预警"""
    if epoch < 5:
        targets = {
            "D_Update_Rate": (0.15, 0.50),
            "G_Success_Rate": (0.05, 0.25),
            "D_Real_Acc": (0.75, 0.95),
            "D_Fake_Acc": (0.65, 0.90),
        }
    else:
        targets = {
            "D_Update_Rate": (0.20, 0.40),
            "G_Success_Rate": (0.10, 0.30),
            "D_Real_Acc": (0.80, 0.92),
            "D_Fake_Acc": (0.70, 0.88),
        }

    print(f"\n{'=' * 80}")
    print(f"🔍 Training Health Check (Epoch {epoch + 1})")
    print(f"{'=' * 80}")

    healthy = True
    for key, (min_val, max_val) in targets.items():
        value = metrics.get(key.lower(), 0)
        status = "✅" if min_val <= value <= max_val else "❌"
        print(f"{status} {key}: {value:.2%} (target: {min_val:.2%}-{max_val:.2%})")
        if status == "❌":
            healthy = False

    if not healthy:
        print("\n⚠️  Training may be unbalanced!")
        if metrics.get("g_success_ratio", 0) < 0.03 and epoch >= 5:
            print("🚨 CRITICAL: G Success Rate too low!")
            print("   Suggested fix:")
            print("   1. Reduce D LR: optimizer_d.param_groups[0]['lr'] *= 0.5")
            print("   2. Increase beta: config.beta *= 1.5")
    else:
        print("\n✅ Training is healthy!")

    print(f"{'=' * 80}\n")
    return healthy


best_content_loss = float("inf")
best_total_loss = float("inf")

for epoch in range(start_gan_epoch, config.train_epochs):
    print(f"\n{'=' * 80}")
    print(f"📊 GAN Epoch [{epoch + 1}/{config.train_epochs}]")
    print(f"Generator LR: {optimizer_g.param_groups[0]['lr']:.6f}")
    print(f"Discriminator LR: {optimizer_d.param_groups[0]['lr']:.6f}")
    print(f"Current Beta: {config.beta:.5f}")
    print(f"{'=' * 80}\n")

    metrics = train_gan(
        train_loader=train_loader,
        generator=generator,
        discriminator=discriminator,
        truncated_vgg19=truncated_vgg19,
        content_loss_criterion=content_loss_criterion,
        adversarial_loss_criterion=adversarial_loss_criterion,
        optimizer_g=optimizer_g,
        optimizer_d=optimizer_d,
        epoch=epoch,
        device=config.device,
        config=config,
        ema=ema,
    )

    scheduler_g.step()
    scheduler_d.step()

    # 🔥 FIX 7: 调用健康监控
    monitor_training_health(epoch, metrics)

    # 🔥 FIX 8: 自动干预机制
    if epoch >= 5 and metrics["g_success_ratio"] < 0.03:
        print("🚨 Automatic intervention triggered!")
        old_d_lr = optimizer_d.param_groups[0]["lr"]
        old_beta = config.beta

        optimizer_d.param_groups[0]["lr"] *= 0.5
        config.beta *= 1.5

        print(f"   D LR: {old_d_lr:.2e} → {optimizer_d.param_groups[0]['lr']:.2e}")
        print(f"   Beta: {old_beta:.5f} → {config.beta:.5f}\n")

    if metrics["loss_content"] < best_content_loss:
        best_content_loss = metrics["loss_content"]
        print(f"🏆 New best content loss: {best_content_loss:.6f}\n")

        save_dict = {
            "gan_epoch": epoch + 1,
            "generator": generator,
            "discriminator": discriminator,
            "optimizer_g": optimizer_g,
            "optimizer_d": optimizer_d,
            "best_content_loss": best_content_loss,
            "metrics": metrics,
            "config": config,  # 🔥 保存config以便恢复超参数
        }

        if ema is not None:
            save_dict["ema_shadow"] = ema.shadow

        save_checkpoint(
            save_dict, filename="checkpoints/checkpoint_srgan_best_content.pth.tar"
        )

    if metrics["loss_g_total"] < best_total_loss:
        best_total_loss = metrics["loss_g_total"]
        print(f"🏆 New best total loss: {best_total_loss:.6f}\n")

        save_dict = {
            "gan_epoch": epoch + 1,
            "generator": generator,
            "discriminator": discriminator,
            "optimizer_g": optimizer_g,
            "optimizer_d": optimizer_d,
            "best_total_loss": best_total_loss,
            "metrics": metrics,
            "config": config,
        }

        if ema is not None:
            save_dict["ema_shadow"] = ema.shadow

        save_checkpoint(save_dict, filename="checkpoints/checkpoint_srgan_best.pth.tar")

        if ema is not None:
            ema.apply_shadow()
            torch.save(generator.state_dict(), "checkpoints/srgan_best_ema.pth")
            ema.restore()
            print("💾 EMA model saved for inference\n")

    if (epoch + 1) % config.save_freq == 0:
        checkpoint_path = f"checkpoints/checkpoint_srgan_epoch{epoch + 1}.pth.tar"

        save_dict = {
            "gan_epoch": epoch + 1,
            "generator": generator,
            "discriminator": discriminator,
            "optimizer_g": optimizer_g,
            "optimizer_d": optimizer_d,
            "metrics": metrics,
            "config": config,
        }

        if ema is not None:
            save_dict["ema_shadow"] = ema.shadow

        save_checkpoint(save_dict, filename=checkpoint_path)
        print(f"💾 Checkpoint saved: {checkpoint_path}\n")

print("\n" + "=" * 80)
print("💾 SAVING FINAL CHECKPOINT")
print("=" * 80)

save_dict = {
    "gan_epoch": config.train_epochs,
    "generator": generator,
    "discriminator": discriminator,
    "optimizer_g": optimizer_g,
    "optimizer_d": optimizer_d,
    "config": config,
}

if ema is not None:
    save_dict["ema_shadow"] = ema.shadow

save_checkpoint(save_dict, filename="checkpoints/checkpoint_srgan_final.pth.tar")

if ema is not None:
    ema.apply_shadow()
    torch.save(generator.state_dict(), "checkpoints/srgan_final_ema.pth")
    ema.restore()
    print("💾 Final EMA model saved")

print(f"\n{'=' * 80}")
print("🎉 TRAINING COMPLETE!")
print(f"{'=' * 80}")
print(f"🏆 Best Content Loss: {best_content_loss:.6f}")
print(f"🏆 Best Total Loss: {best_total_loss:.6f}")
print(f"{'=' * 80}\n")